<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Fourier-Transform-of-the-Lorentzian-Weighted-Fermi-Dirac-function" data-toc-modified-id="Fourier-Transform-of-the-Lorentzian-Weighted-Fermi-Dirac-function-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Fourier Transform of the Lorentzian-Weighted Fermi-Dirac function</a></span><ul class="toc-item"><li><span><a href="#Integral-to-evaluate" data-toc-modified-id="Integral-to-evaluate-1.1"><span class="toc-item-num">1.1&nbsp;&nbsp;</span>Integral to evaluate</a></span></li><li><span><a href="#Analytical-result" data-toc-modified-id="Analytical-result-1.2"><span class="toc-item-num">1.2&nbsp;&nbsp;</span>Analytical result</a></span></li><li><span><a href="#Numerical-verification" data-toc-modified-id="Numerical-verification-1.3"><span class="toc-item-num">1.3&nbsp;&nbsp;</span>Numerical verification</a></span></li></ul></li></ul></div>

In [1]:
import numpy as np
import mpmath as mp
from scipy.integrate import quad

rng = np.random.default_rng(seed=42)

## Fourier Transform of the Lorentzian-Weighted Fermi-Dirac function

### Integral to evaluate

Let $z_a = \epsilon_a-i\eta$ and $z_b = \epsilon_b+i\eta$.
We want to evaluate the integral
$$
I(t_r) = \int_{-\infty}^{\infty} dE \,
e^{-iEt_r/\hbar} f(E)
\frac{\eta}{(E-z_a)(E-z_b)},
$$
where the Fermi-Dirac distribution is
$$
f(E)=\frac{1}{1+e^{\beta(E-\mu)}}.
$$
We assume $\eta>0$, $\beta>0$, and real $\epsilon_a,\epsilon_b,\mu,t_r$.

### Analytical result

For $t_r\neq 0$, the analytical results is
$$
\boxed{
\begin{aligned}
I(t_r)
=\frac{\eta}{z_b-z_a}
\bigg(&
2\pi i\,e^{-iz_st_r/\hbar}f(z_s) \\
&- e^{-i\mu t_r/\hbar}
\sqrt{q}
\left[
\Phi(q,1,v_b^{(s)}) - \Phi(q,1,v_a^{(s)})
\right]\bigg).
\end{aligned}
}
$$

Here, we define
$$
\begin{aligned}
s &= \operatorname{sgn}(t_r) , \\
z_s &= \frac{(1+s) z_a + (1-s) z_b}{2} , \\
q &= e^{-2\pi|t_r|/(\beta\hbar)} , \\
v_j^{(s)} &= \frac{1}{2} +
\frac{i s \beta(\mu-z_j)}{2\pi}, \qquad j=a,b .
\end{aligned}
$$

The function $\Phi(q,1,v)$ is the **Lerch transcendent**,
$$
\Phi(q,1,v)=
\sum_{n=0}^{\infty}\frac{q^n}{n+v} .
$$

### Numerical verification

We evaluate $I(t_r)$ in two independent ways:
- **Numerical integration:** Direct quadrature over the real
  energy axis using `scipy.integrate.quad`.
- **Analytical expression:** Evaluation of the closed-form
  result using `mpmath.lerchphi`.

We compare the complex-valued results through the absolute error
$$
\mathrm{Error}(t_r)=
\left|
I_{\mathrm{numerical}}(t_r)
- I_{\mathrm{analytical}}(t_r)
\right|.
$$

In [2]:
# Parameters
beta = rng.uniform(0,3)
mu = rng.uniform(0,3)
eta = rng.uniform(0,0.3)
epsilon_a = rng.uniform(-3,3)
epsilon_b = rng.uniform(-3,3)

print(f"beta: {beta:.3f}")
print(f"mu: {mu:.3f}")
print(f"eta: {eta:.3f}")
print(f"epsilon_a: {epsilon_a:.3f}")
print(f"epsilon_b: {epsilon_b:.3f}")

za = epsilon_a - 1j * eta
zb = epsilon_b + 1j * eta

beta: 2.322
mu: 1.317
eta: 0.258
epsilon_a: 1.184
epsilon_b: -2.435


In [3]:
# Fermi function
def f(E):
    return 1 / (1 + mp.exp(beta * (E - mu)))

# integrand
def g(E):
    return eta * f(E) / ((E - za) * (E - zb))

In [4]:
# decompose complex exponential as cos and sin
# integrate real and imaginary part separately
# using SciPy's weighted oscillatory quadrature

def integrate_plain(func):
    return (
        quad(func, -np.inf, mu, epsabs=1e-11)[0]
        + quad(func, mu, np.inf, epsabs=1e-11)[0]
    )

def integrate_weighted(func, weight, wvar):
    return (
        quad(
            func, -np.inf, mu, epsabs=1e-11, 
            weight=weight, wvar=wvar, limit=500
        )[0]
        + quad(
            func, mu, np.inf, epsabs=1e-11, 
            weight=weight, wvar=wvar, limit=500
        )[0]
    )


def integral_numeric(tr):
    if tr == 0:
        real_part = integrate_plain(lambda E: g(E).real)
        imag_part = integrate_plain(lambda E: g(E).imag)
    else:
        real_part = (
            integrate_weighted(lambda E: g(E).real, 'cos', tr)
            + integrate_weighted(lambda E: g(E).imag, 'sin', tr)
        )

        imag_part = (
            integrate_weighted(lambda E: g(E).imag, 'cos', tr)
            - integrate_weighted(lambda E: g(E).real, 'sin', tr)
        )

    return real_part + 1j * imag_part

In [5]:
# Analytical Lerch-transcendent expression
def integral_lerch(tr):
    if tr == 0:
        raise ValueError("Use the digamma formula at tr=0.")

    s = np.sign(tr)

    q = mp.exp(-2 * mp.pi * abs(tr) / beta)

    va = 0.5 + 1j * s * beta * (mu - za) / (2 * mp.pi)
    vb = 0.5 + 1j * s * beta * (mu - zb) / (2 * mp.pi)

    zs = ((1+s) * za + (1-s) * zb) / 2

    pole_term = (
        2j * mp.pi
        * mp.exp(-1j * zs * tr)
        * f(zs)
    )

    lerch_term = (
        mp.exp(-1j * mu * tr)
        * mp.exp(-mp.pi * abs(tr) / beta)
        * (mp.lerchphi(q,1,vb) - mp.lerchphi(q,1,va))
    )

    return complex(eta / (zb-za) * (pole_term - lerch_term))

In [6]:
# Comparison (include both negative and positive tr)
for tr in [-5, -1, -0.1, 0.1, 1, 5]: 
    numerical = integral_numeric(tr)
    analytical = integral_lerch(tr)

    print(f"tr = {tr:+.3f}")
    print(f"  Numerical  = {numerical}")
    print(f"  Analytical = {analytical}")
    print(f"  Error      = {abs(numerical - analytical):.3e}")
    print()

tr = -5.000
  Numerical  = (0.061917801696532104-0.1051828702199315j)
  Analytical = (0.06191780169653385-0.1051828702199374j)
  Error      = 6.152e-15

tr = -1.000
  Numerical  = (-0.2696064945352553+0.20044678312413708j)
  Analytical = (-0.26960649453525665+0.20044678312414105j)
  Error      = 4.187e-15

tr = -0.100
  Numerical  = (-0.16317944942868357-0.400120624053485j)
  Analytical = (-0.16317944942868096-0.4001206240534893j)
  Error      = 5.055e-15

tr = +0.100
  Numerical  = (-0.08605248656441508-0.2950497162889893j)
  Analytical = (-0.0860524865644109-0.2950497162889804j)
  Error      = 9.815e-15

tr = +1.000
  Numerical  = (-0.16757180303075014-0.1103861101807371j)
  Analytical = (-0.16757180303075347-0.11038611018073535j)
  Error      = 3.762e-15

tr = +5.000
  Numerical  = (0.050264959327859945-0.05354236131122248j)
  Analytical = (0.05026495932786246-0.05354236131121664j)
  Error      = 6.360e-15

